# Ophyd electrochemistry quickstart

This notebook runs the intended Ophyd/Bluesky workflow entirely against the deterministic simulator. It does not import PyVISA, discover instruments or communicate with hardware. It demonstrates immutable experiment configuration, the Flyer lifecycle, paired I/V events and a preliminary post-run visualization.

This is a living quickstart. Review it whenever the public request model, measurement/status schema, device lifecycle, target backend or commissioned capabilities change. The real 2460 adapter remains unavailable until the START/aperture timestamp relationship is established.

In [ ]:
from __future__ import annotations

import threading
import time

from bluesky import RunEngine
from event_model import unpack_event_page

from examples.compile_program import capabilities, config
from examples.follower import pulse_train_acquisition
from ophyd_electrochemistry import (
    AcquisitionRequest,
    CurrentPulseSequence,
    DeviceState,
    StartMode,
)
from ophyd_electrochemistry.keithley.k2460 import (
    Keithley2460Compiler,
    Keithley2460Device,
)
from ophyd_electrochemistry.simulation import FakeCell, FakeTransport, SimulatedRuntime

## 1. Define one finite request

The imported `config` and `capabilities` are explicitly synthetic fixtures. They demonstrate the commissioned-configuration layer; they are not Keithley specifications or safe battery defaults. The user supplies the per-run scientific intent below as one immutable request.

In [ ]:
program = CurrentPulseSequence(
    baseline_current_a=0.0,
    pulse_current_a=0.01,
    pulse_width_s=0.020,
    period_s=0.050,
    count=4,
    sample_period_s=0.010,
    voltage_limit_v=5.0,
)
request = AcquisitionRequest(
    experiment_id="quickstart-synthetic-pulse-train",
    start_mode=StartMode.IMMEDIATE,
    program=program,
)
compiled = Keithley2460Compiler(capabilities).compile(request, config)
{
    "hardware_ready": compiled.hardware_ready,
    "duration_s": compiled.achieved_duration_s,
    "measurement_records": compiled.budget.measurement_records,
    "request_sha256": compiled.request_sha256,
    "program_sha256": compiled.program_sha256,
}

## 2. Construct and run the device

`Keithley2460Device` consumes a typed backend. Here that backend is `FakeTransport` over `SimulatedRuntime`. The helper thread advances only the simulator's virtual clock; source timing remains runtime-owned rather than host-paced. The plan performs stage, prepare, kickoff, complete, collect and unstage.

In [ ]:
runtime = SimulatedRuntime(
    capabilities=capabilities,
    config=config,
    cell=FakeCell(resistance_ohm=10.0, open_circuit_voltage_v=0.1),
)
backend = FakeTransport(runtime)
ec = Keithley2460Device(
    backend,
    name="ec",
    poll_period_s=0.001,
    completion_timeout_s=2.0,
    shutdown_timeout_s=1.0,
    collection_chunk_records=7,
)

documents = []
run_engine = RunEngine({})
run_engine.subscribe(lambda name, document: documents.append((name, document)))
worker_errors = []


def finish_simulated_program():
    try:
        deadline = time.monotonic() + 2.0
        while ec.state != DeviceState.RUNNING and time.monotonic() < deadline:
            time.sleep(0.001)
        if ec.state != DeviceState.RUNNING:
            raise TimeoutError("Simulator did not enter RUNNING")
        runtime.advance_ticks(1_000)
    except BaseException as exc:
        worker_errors.append(exc)


worker = threading.Thread(target=finish_simulated_program, daemon=True)
worker.start()
run_uids = run_engine(pulse_train_acquisition(ec, request))
worker.join(timeout=2.0)
assert not worker.is_alive() and not worker_errors
assert ec.state == DeviceState.COMPLETE and ec.output_enabled is False
{"run_uids": run_uids, "final_state": ec.state.value, "output": ec.output_enabled}

## 3. Inspect paired I/V records

The `electrochemistry` stream contains one event per complete paired sample. `ec_current` and `ec_voltage` therefore share the same aperture and mapping metadata. Source and measurement status remain separate untouched words. In this simulator, source-status bit 0 means synthetic compliance was active during at least part of the aperture; measurement status is currently zero. These are not Keithley bit definitions.

In [ ]:
events = []
for name, document in documents:
    if name == "event":
        events.append(document)
    elif name == "event_page":
        events.extend(unpack_event_page(document))

assert events
rows = [event["data"] for event in events]
columns = (
    "ec_sample_index",
    "ec_time_relative",
    "ec_voltage",
    "ec_current",
    "ec_source_setpoint",
    "ec_mapping_quality",
    "ec_source_status",
    "ec_measurement_status",
)
print(" | ".join(columns))
print("-" * 118)
for row in rows[:12]:
    print(
        f"{row['ec_sample_index']:>3d} | "
        f"{row['ec_time_relative']:>8.4f} | "
        f"{row['ec_voltage']:>9.6f} | "
        f"{row['ec_current']:>9.6f} | "
        f"{row['ec_source_setpoint']:>9.6f} | "
        f"{row['ec_mapping_quality']:<22s} | "
        f"{row['ec_source_status']} | "
        f"{row['ec_measurement_status']}"
    )
print(f"\n{len(rows)} total paired records; table truncated to 12 rows.")

## 4. Preliminary visualization

The next cell creates an inline SVG using only the standard library plus the display facility already provided by Jupyter. Shaded regions are commanded pulse dwells. This is a first-look diagnostic, not calibration, timing-acceptance or settling evidence.

In [ ]:
width, height = 900, 500
left, right = 80, 25
top_voltage, bottom_voltage = 45, 225
top_current, bottom_current = 285, 465
plot_width = width - left - right
times = [row["ec_time_relative"] for row in rows]
voltages = [row["ec_voltage"] for row in rows]
currents = [row["ec_current"] for row in rows]
time_max = max(compiled.achieved_duration_s, max(times))


def x_position(value):
    return left + plot_width * value / time_max


def y_position(value, values, top, bottom):
    low, high = min(values), max(values)
    padding = max((high - low) * 0.1, 1e-12)
    low, high = low - padding, high + padding
    return bottom - (bottom - top) * (value - low) / (high - low)


def polyline(values, top, bottom, color):
    points = " ".join(
        f"{x_position(t):.2f},{y_position(v, values, top, bottom):.2f}"
        for t, v in zip(times, values, strict=True)
    )
    return f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="2"/>'


parts = [
    f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}">',
    '<rect width="100%" height="100%" fill="white"/>',
]
for pulse_index in range(program.count):
    pulse_start = pulse_index * program.period_s
    pulse_end = pulse_start + program.pulse_width_s
    x0, x1 = x_position(pulse_start), x_position(pulse_end)
    for top, bottom in ((top_voltage, bottom_voltage), (top_current, bottom_current)):
        parts.append(
            f'<rect x="{x0:.2f}" y="{top}" width="{x1 - x0:.2f}" '
            f'height="{bottom - top}" fill="#e8eef8"/>'
        )
for top, bottom in ((top_voltage, bottom_voltage), (top_current, bottom_current)):
    parts.append(
        f'<rect x="{left}" y="{top}" width="{plot_width}" '
        f'height="{bottom - top}" fill="none" stroke="#555"/>'
    )
parts.extend(
    (
        polyline(voltages, top_voltage, bottom_voltage, "#1565c0"),
        polyline(currents, top_current, bottom_current, "#c62828"),
        '<text x="20" y="35" font-size="16" fill="#1565c0">Voltage (V)</text>',
        '<text x="20" y="275" font-size="16" fill="#c62828">Current (A)</text>',
        f'<text x="{width / 2 - 55}" y="495" font-size="15">Time from START (s)</text>',
        "</svg>",
    )
)
svg = "".join(parts)
try:
    from IPython.display import SVG, display
except ModuleNotFoundError:
    print(svg)
else:
    display(SVG(svg))

## What changes for hardware?

The request, plan lifecycle, stream name, paired-record concept and separate `ec_source_status`/`ec_measurement_status` fields remain the same. A commissioned target backend replaces `FakeTransport`; startup owns the VISA resource, terminals/sense, digital I/O, safety envelope, timing requirements and buffer limits. Before that backend is enabled, the timestamp proof must establish actual-START-relative aperture meaning and the target adapter must supply the documented Keithley definitions for both status words.